# 06_Qdrant_Index — Indexation vectorielle hybride

Une seule collection `entraide_ma`, vecteurs **nommés** dense (BGE-M3,
1024D, cosine) + sparse (TF-IDF hashé, précalculé en 04_Embed) — corrige le
bug v1 (deux collections séparées, dont une sparse jamais réellement
peuplée). Le **payload est enrichi** par jointure sur `entity_id` avec
`data/processed/*.jsonl`, condition nécessaire pour que `context_builder`
côté application reçoive autre chose qu'un texte brut (nom, région, adresse,
conditions, documents, institutions, budget...).

In [1]:
import sys, os, hashlib
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

sys.path.insert(0, str(Path.cwd()))
from etl_lib.io_utils import load_jsonl

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
EMBEDDINGS_DIR = Path.cwd().parent / "data" / "embeddings"

embeddings = load_jsonl(EMBEDDINGS_DIR / "embeddings_all.jsonl")
services = {s["id"]: s for s in load_jsonl(PROCESSED_DIR / "services_unified.jsonl")}
centres = {c["id"]: c for c in load_jsonl(PROCESSED_DIR / "centres_normalized.jsonl")}
programmes = {p["id"]: p for p in load_jsonl(PROCESSED_DIR / "programmes_2027.jsonl")}
faqs = {f["id"]: f for f in load_jsonl(PROCESSED_DIR / "faq_aos.jsonl")}

# === FAQ_EXTRA_INGESTION_BLOCK ===
# Fusion avec les FAQ issues de data_faq (memes cles question_ar/reponse_ar
# -> la branche FAQ de enrich_payload() plus bas fonctionne sans changement)
# + profils Institution (nouvelle branche ajoutee plus bas).
faqs.update({f["id"]: f for f in load_jsonl(PROCESSED_DIR / "faq_extra_service.jsonl")})
faqs.update({f["id"]: f for f in load_jsonl(PROCESSED_DIR / "faq_extra_general.jsonl")})
institution_profiles = {ip["institution_code"]: ip for ip in load_jsonl(PROCESSED_DIR / "institution_enrich.jsonl")}

print(f"embeddings={len(embeddings)} services={len(services)} centres={len(centres)} "
      f"programmes={len(programmes)} faq={len(faqs)} institution_profiles={len(institution_profiles)}")

embeddings=10491 services=61 centres=3328 programmes=7 faq=121 institution_profiles=8


## 1. Enrichissement du payload par jointure sur `entity_id`

In [2]:
def enrich_payload(base: dict) -> dict:
    entity_type, entity_id = base.get("entity_type"), base.get("entity_id")
    enriched = dict(base)

    if entity_type == "Service" and entity_id in services:
        s = services[entity_id]
        enriched.update({
            "name": s["service_ar"], "conditions": s["conditions_ar"], "documents": s["documents_ar"],
            "is_eps": s["eps_fallback"],
        })
    elif entity_type == "Centre" and entity_id in centres:
        c = centres[entity_id]
        enriched.update({
            "name": c["nom"], "region": c["region"], "delegation": c["delegation"], "commune": c["commune"],
            "address": c["adresse"], "milieu": c["milieu"], "capacite": c["capacite"],
            "is_eps": "EPS" in c["institutions"] and len(c["institutions"]) == 1,
        })
    elif entity_type == "Programme" and entity_id in programmes:
        p = programmes[entity_id]
        enriched.update({"name": p["titre_fr"] or p["titre_ar"], "budget": p.get("budget"), "is_eps": False})
    elif entity_type == "FAQ" and entity_id in faqs:
        f = faqs[entity_id]
        enriched.update({"name": f["question_ar"][:120], "question": f["question_ar"],
                          "reponse": f["reponse_ar"], "is_eps": False})
    return enriched


sample = enrich_payload(embeddings[0]["payload"])
print("Exemple de payload enrichi:", {k: v for k, v in sample.items() if k != "text"})


Exemple de payload enrichi: {'entity_type': 'Service', 'entity_id': 'svc_98978a9e9dac', 'type': 'service', 'chunk_type': 'description', 'langue': 'ar', 'population_cible': 'enfants', 'institutions': ['CAPE', 'UPE'], 'name': 'المساعدة الاجتماعية (الاستماع والإرشاد والوساطة والتوجيه والمواكبة)', 'conditions': ['الأطفال في وضعية صعبة أو وضعية هشاشة'], 'documents': ['بطافة التعريف الوطنية لولي الأمر - نسخة من عقد الازدياد أو وثيقة إدارية تثبت هوية الطفل- الإعفاء من أي وثيقة بالنسبة للأطفال في وضعية الشارع', 'بطافة التعريف الوطنية لولي الأمر', 'نسخة من عقد الازدياد أو وثيقة إدارية تثبت هوية الطفل', 'الإعفاء من أي وثيقة بالنسبة للأطفال في وضعية الشارع'], 'is_eps': False}


In [3]:
# === FAQ_EXTRA_INGESTION_BLOCK ===
def enrich_payload_institution(base: dict) -> dict:
    """Meme role que enrich_payload() pour entity_type == 'Institution' --
    fonction separee pour ne pas alourdir la cascade if/elif existante avec
    un cas qui n'existe pas encore dans le schema d'origine du guide."""
    entity_id = base.get("entity_id")
    enriched = dict(base)
    if entity_id in institution_profiles:
        ip = institution_profiles[entity_id]
        enriched.update({"name": entity_id, "question": ip["question_ar"], "is_eps": False})
    return enriched


## 2. Création de la collection (dense + sparse nommés)

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, PayloadSchemaType, PointStruct, SparseVector,
)

QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", 6333))
COLLECTION = os.getenv("QDRANT_COLLECTION", "entraide_mvp")
DENSE_NAME, SPARSE_NAME = "dense", "sparse"

client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={DENSE_NAME: VectorParams(size=len(embeddings[0]["vector"]), distance=Distance.COSINE)},
    sparse_vectors_config={SPARSE_NAME: SparseVectorParams()},
)
for field, schema in {
    "entity_type": PayloadSchemaType.KEYWORD, "entity_id": PayloadSchemaType.KEYWORD,
    "chunk_type": PayloadSchemaType.KEYWORD, "region": PayloadSchemaType.KEYWORD,
    "langue": PayloadSchemaType.KEYWORD, "is_eps": PayloadSchemaType.BOOL,
}.items():
    client.create_payload_index(COLLECTION, field_name=field, field_schema=schema)

print(f"Collection '{COLLECTION}' creee (dense={len(embeddings[0]['vector'])}D, sparse=hashe)")


/home/ubunto/entraide-mvp/.venv/lib/python3.14/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


Collection 'entraide_mvp' creee (dense=1024D, sparse=hashe)


## 3. Indexation par lots

In [5]:
def stable_point_id(chunk_id: str) -> int:
    return int(hashlib.md5(chunk_id.encode("utf-8")).hexdigest()[:15], 16)


BATCH_SIZE = 500
points = []
for e in embeddings:
    # === FAQ_EXTRA_INGESTION_BLOCK ===
    entity_type = e["payload"].get("entity_type")
    enriched = enrich_payload_institution(e["payload"]) if entity_type == "Institution" else enrich_payload(e["payload"])
    points.append(PointStruct(
        id=stable_point_id(e["id"]),
        vector={
            DENSE_NAME: e["vector"],
            SPARSE_NAME: SparseVector(indices=e.get("sparse_indices", []), values=e.get("sparse_values", [])),
        },
        payload=enriched,
    ))

for i in range(0, len(points), BATCH_SIZE):
    batch = points[i:i + BATCH_SIZE]
    client.upsert(collection_name=COLLECTION, points=batch)
    print(f"  {min(i + BATCH_SIZE, len(points))}/{len(points)} points indexes")

info = client.get_collection(COLLECTION)
print(f"\nPoints dans la collection : {info.points_count}")


  500/10491 points indexes


  1000/10491 points indexes


  1500/10491 points indexes


  2000/10491 points indexes


  2500/10491 points indexes


...

  8500/10491 points indexes


  9000/10491 points indexes


  9500/10491 points indexes


  10000/10491 points indexes


  10491/10491 points indexes

Points dans la collection : 10491


## 4. Vérification — cohérence avec Neo4j

In [6]:
all_entity_ids_qdrant = set()
offset = None
while True:
    pts, offset = client.scroll(COLLECTION, limit=1000, offset=offset, with_payload=["entity_id"], with_vectors=False)
    all_entity_ids_qdrant.update(p.payload.get("entity_id") for p in pts if p.payload.get("entity_id"))
    if offset is None:
        break

all_source_ids = set(services) | set(centres) | set(programmes) | set(faqs)
coherence = 100 * len(all_source_ids & all_entity_ids_qdrant) / max(len(all_source_ids), 1)
print(f"IDs sources (services+centres+programmes+faq) : {len(all_source_ids)}")
print(f"IDs distincts trouvables dans Qdrant           : {len(all_entity_ids_qdrant)}")
print(f"Coherence : {coherence:.1f}% (cible guide : 100%)")

print("\n\u2705 06_Qdrant_Index termine. Pipeline 00->06 complet.")


IDs sources (services+centres+programmes+faq) : 3517
IDs distincts trouvables dans Qdrant           : 3525
Coherence : 100.0% (cible guide : 100%)

✅ 06_Qdrant_Index termine. Pipeline 00->06 complet.
